In [ ]:
import xml.etree.ElementTree as ET
import spacy

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Your XML data
xml_data = '''<?xml version="1.0" encoding="UTF-8"?>
<ns0:Order xmlns:ns0="http://example.com/orders" orderID="ORD12345" status="Processing">
    <Customer customerID="CUST67890">
        <Name>John Doe</Name>
        <Contact>
            <Email>john.doe@example.com</Email>
            <Phone type="mobile">+1-234-567-8901</Phone>
        </Contact>
        <Address type="Shipping">
            <Street>123 Elm Street</Street>
            <City>Metropolis</City>
            <State>NY</State>
            <PostalCode>10101</PostalCode>
            <Country code="US">United States</Country>
        </Address>
    </Customer>
    <Items>
        <Item SKU="PROD001" quantity="2">
            <ProductName>Wireless Mouse</ProductName>
            <Price currency="USD">25.99</Price>
        </Item>
    </Items>
    <Payment method="CreditCard">
        <TransactionID>TXN998877</TransactionID>
        <Amount currency="USD">131.97</Amount>
        <Status>Authorized</Status>
    </Payment>
    <Notes>
        <![CDATA[
            Please deliver between 9 AM - 5 PM. Leave at the front door if no one is home.
        ]]>
    </Notes>
    <Audit>
        <CreatedBy>system</CreatedBy>
        <CreatedOn>2025-05-06T10:15:00Z</CreatedOn>
    </Audit>
</ns0:Order>
'''

# Register namespace
ns = {'ns0': 'http://example.com/orders'}
ET.register_namespace('', 'http://example.com/orders')
root = ET.fromstring(xml_data)

# Find elements using the correct path with namespace
customer = root.find('ns0:Customer', ns)
name = customer.find('ns0:Name', ns).text if customer is not None else ''
address = customer.find('ns0:Address', ns) if customer is not None else None
street = address.find('ns0:Street', ns).text if address is not None else ''
city = address.find('ns0:City', ns).text if address is not None else ''
country = address.find('ns0:Country', ns).text if address is not None else ''
notes = root.find('ns0:Notes', ns).text.strip() if root.find('ns0:Notes', ns) is not None else ''

# Combine text for NLP
text_for_nlp = f"{name} lives at {street}, {city}, {country}. {notes}"

# Apply NLP
doc = nlp(text_for_nlp)

# Output entities
print(f"Analyzing: {text_for_nlp}\n")
print("Named Entities:")
for ent in doc.ents:
    print(f" - {ent.text} ({ent.label_})")


Analyzing:  lives at , , . 

Named Entities:


XML SCrapping and Text Summerization Using NLP

In [ ]:
import xml.etree.ElementTree as ET
from transformers import pipeline

# Load summarization pipeline (uses a pretrained model like BART or T5)
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# Sample XML string
xml_data = '''<orders>
    <order id="1001">
        <customer>
            <name>Jane Smith</name>
        </customer>
        <note>
            <![CDATA[
                Please deliver the package after 6 PM. If nobody is home, kindly leave it with the neighbor at 458 Oak Avenue. The front gate is usually locked.
            ]]>
        </note>
    </order>
    <order id="1002">
        <customer>
            <name>Mark Johnson</name>
        </customer>
        <note>
            <![CDATA[
                Ring the doorbell twice and wait for at least one minute before leaving. Do not leave the package outside as it might get stolen.
            ]]>
        </note>
    </order>
</orders>
'''  # Use full XML above

# Parse XML
root = ET.fromstring(xml_data)

# Loop through orders
for order in root.findall('order'):
    customer_name = order.find('customer/name').text
    note = order.find('note').text.strip()

    print(f"\n🧾 Customer: {customer_name}")
    print(f"📝 Original Note:\n{note}")

    # Summarize note (shorten if too long)
    summary = summarizer(note, max_length=40, min_length=15, do_sample=False)[0]['summary_text']
    print(f"🧠 Summarized Note:\n{summary}")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0
Your max_length is set to 40, but your input_length is only 33. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=16)



🧾 Customer: Jane Smith
📝 Original Note:
Please deliver the package after 6 PM. If nobody is home, kindly leave it with the neighbor at 458 Oak Avenue. The front gate is usually locked.


Your max_length is set to 40, but your input_length is only 29. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=14)


🧠 Summarized Note:
If nobody is home, kindly leave it with the neighbor at 458 Oak Avenue. The front gate is usually locked. Please deliver the package after 6 PM.

🧾 Customer: Mark Johnson
📝 Original Note:
Ring the doorbell twice and wait for at least one minute before leaving. Do not leave the package outside as it might get stolen.
🧠 Summarized Note:
Ring the doorbell twice and wait for at least one minute before leaving. Do not leave the package outside as it might get stolen.


JSON Scraping + Summarization

In [ ]:
import json
from transformers import pipeline

# Load summarization pipeline (BART works well)
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# JSON data (normally you'd load this from a file or API)
json_data = [
    {
        "order_id": "1001",
        "customer": { "name": "Jane Smith" },
        "note": "Please deliver the package after 6 PM. If nobody is home, kindly leave it with the neighbor at 458 Oak Avenue. The front gate is usually locked."
    },
    {
        "order_id": "1002",
        "customer": { "name": "Mark Johnson" },
        "note": "Ring the doorbell twice and wait for at least one minute before leaving. Do not leave the package outside as it might get stolen."
    }
]

# Process each order
for order in json_data:
    name = order['customer']['name']
    note = order['note']

    print(f"\n🧾 Customer: {name}")
    print(f"📝 Original Note:\n{note}")

    # Summarize using NLP
    summary = summarizer(note, max_length=40, min_length=15, do_sample=False)[0]['summary_text']
    print(f"🧠 Summarized Note:\n{summary}")


Device set to use cuda:0
Your max_length is set to 40, but your input_length is only 33. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=16)



🧾 Customer: Jane Smith
📝 Original Note:
Please deliver the package after 6 PM. If nobody is home, kindly leave it with the neighbor at 458 Oak Avenue. The front gate is usually locked.


Your max_length is set to 40, but your input_length is only 29. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=14)


🧠 Summarized Note:
If nobody is home, kindly leave it with the neighbor at 458 Oak Avenue. The front gate is usually locked. Please deliver the package after 6 PM.

🧾 Customer: Mark Johnson
📝 Original Note:
Ring the doorbell twice and wait for at least one minute before leaving. Do not leave the package outside as it might get stolen.
🧠 Summarized Note:
Ring the doorbell twice and wait for at least one minute before leaving. Do not leave the package outside as it might get stolen.
